# So sánh DETR gốc và DETR + skip trên RefCOCO testA

Chấm hai checkpoint best trên cùng các câu thuộc `testA` trong `person_samples.json`.
Nhóm màu là tập con chứa ít nhất một từ nguyên vẹn trong danh sách
`red, blue, white, black, green, yellow`, khớp cách lọc trong notebook skip.

Bật GPU trên Kaggle. Điền hai đường dẫn checkpoint trong ô cấu hình rồi chạy Run All.
Notebook tìm annotation, ảnh và `detr-r50-unc.pth` trong Kaggle Input. Nếu thiếu dữ liệu hoặc trọng số phụ, notebook thử tải qua mạng. RoBERTa cũng cần mạng nếu phiên chạy chưa có cache Hugging Face.

In [ ]:
%pip install -q "transformers==4.47.1" "gdown==5.2.0" kagglehub

In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import math
import os
import re

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.ops import box_convert
from transformers import AutoTokenizer, AutoModel, AutoConfig
from tqdm.auto import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"

## Cấu hình Kaggle

In [ ]:
INPUT_DIR = Path("/kaggle/input")
BASELINE_BEST_PATH = Path("/kaggle/input/DUONG_DAN_DETR_GOC/best_checkpoint.pth")
SKIP_BEST_PATH = Path("/kaggle/input/DUONG_DAN_DETR_SKIP/best_checkpoint.pth")

# Khi có nhiều bản trùng tên trong Input, điền đường dẫn chính xác ở đây.
ANNOTATION_PATH = None
IMAGE_DIR = None
DETR_CHECKPOINT = None

BATCH_SIZE = 16
NUM_WORKERS = 2
OUTPUT_DIR = Path("/kaggle/working/rec_testA_comparison")
COLOR_WORDS = ("red", "blue", "white", "black", "green", "yellow")

In [ ]:
def unique_input_file(name):
    matches = sorted(INPUT_DIR.rglob(name))
    if len(matches) > 1:
        candidates = "\n".join(map(str, matches))
        raise ValueError(f"Có nhiều file {name}. Điền đường dẫn trong ô cấu hình.\n{candidates}")
    return matches[0] if matches else None


checkpoint_paths = {"DETR gốc": BASELINE_BEST_PATH, "DETR + skip": SKIP_BEST_PATH}
missing = [str(path) for path in checkpoint_paths.values() if not path.is_file()]
if missing:
    candidates = "\n".join(map(str, sorted(INPUT_DIR.rglob("*.pth"))[:25]))
    raise FileNotFoundError(
        "Điền đường dẫn hai checkpoint best trong ô cấu hình.\n"
        + "\n".join(missing) + "\nCác file .pth trong Input:\n" + candidates
    )
if BASELINE_BEST_PATH.resolve() == SKIP_BEST_PATH.resolve():
    raise ValueError("Hai model đang trỏ tới cùng một file.")
if not torch.cuda.is_available():
    raise RuntimeError("Bật GPU trong Kaggle Settings rồi chạy lại.")
device = torch.device("cuda:0")
print("GPU:", torch.cuda.get_device_name(device))

if ANNOTATION_PATH is None:
    ANNOTATION_PATH = unique_input_file("person_samples.json")
    if ANNOTATION_PATH is None:
        import kagglehub

        dataset_dir = Path(kagglehub.dataset_download("lamky690/refcoco/versions/1"))
        ANNOTATION_PATH = dataset_dir / "refcoco_data" / "person_samples.json"
ANNOTATION_PATH = Path(ANNOTATION_PATH)
IMAGE_DIR = Path(IMAGE_DIR) if IMAGE_DIR is not None else ANNOTATION_PATH.parent / "images"

if DETR_CHECKPOINT is None:
    DETR_CHECKPOINT = unique_input_file("detr-r50-unc.pth")
    if DETR_CHECKPOINT is None:
        import gdown

        DETR_CHECKPOINT = Path("/kaggle/working/detr-r50-unc.pth")
        if not DETR_CHECKPOINT.is_file():
            partial = DETR_CHECKPOINT.with_suffix(".download")
            result = gdown.download(
                id="1U8Qdl-StW1yaJapnGtAEFO8Ic4X0ubtp", output=str(partial), quiet=False
            )
            if result is None or not partial.is_file():
                raise FileNotFoundError("Gắn detr-r50-unc.pth vào Kaggle Input và điền DETR_CHECKPOINT.")
            partial.replace(DETR_CHECKPOINT)
DETR_CHECKPOINT = Path(DETR_CHECKPOINT)

## Kiểm tra cấu hình và dữ liệu

In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def check_checkpoint(name, payload):
    cfg = payload["config"]
    is_skip = name == "DETR + skip"
    expected_stage = "rec_finetune_layer4_text2_skip" if is_skip else "rec_finetune_layer4_text2"
    expected_alpha = 0.10 if is_skip else 0.0
    if cfg["stage"] != expected_stage or cfg.get("skip_alpha", 0.0) != expected_alpha:
        raise ValueError(f"{name}: sai checkpoint hoặc hệ số skip.")
    if payload["epoch"] != payload["best_epoch"]:
        raise ValueError(f"{name}: chọn checkpoint tại epoch best.")
    if "color_words" in cfg and tuple(cfg["color_words"]) != COLOR_WORDS:
        raise ValueError(f"{name}: danh sách từ màu khác cấu hình đánh giá.")
    return cfg


configs = {}
checkpoint_info = {}
for name, path in checkpoint_paths.items():
    # Chỉ giữ một payload trên CPU mỗi lần. mmap giảm I/O với checkpoint dạng ZIP.
    payload = torch.load(path, map_location="cpu", weights_only=True, mmap=True)
    configs[name] = check_checkpoint(name, payload)
    checkpoint_info[name] = dict(epoch=int(payload["epoch"]), val_acc50=float(payload["best_acc50"]))
    del payload

baseline_cfg, skip_cfg = configs.values()
for key in ("annotation_sha256", "detr_sha256", "text_cache_spec", "image_size_hw",
            "text_model", "text_revision"):
    if baseline_cfg[key] != skip_cfg[key]:
        raise ValueError(f"Hai checkpoint khác cấu hình ở {key}.")

if file_sha256(ANNOTATION_PATH) != skip_cfg["annotation_sha256"]:
    raise ValueError("person_samples.json khác bản dùng khi train.")
if file_sha256(DETR_CHECKPOINT) != skip_cfg["detr_sha256"]:
    raise ValueError("detr-r50-unc.pth khác bản dùng khi train.")

IMAGE_SIZE = tuple(skip_cfg["image_size_hw"])
TEXT_MODEL = skip_cfg["text_model"]
TEXT_REVISION = skip_cfg["text_revision"]
test_samples = [
    row for row in json.loads(ANNOTATION_PATH.read_text(encoding="utf-8"))
    if row["split"] == "testA"
]
color_pattern = re.compile(r"\b(?:" + "|".join(map(re.escape, COLOR_WORDS)) + r")\b", re.IGNORECASE)
color_mask = np.array([bool(color_pattern.search(row["text"])) for row in test_samples])
if not color_mask.any():
    raise ValueError("Không tìm thấy câu màu trong testA.")
for row in test_samples:
    path = IMAGE_DIR / row["file_name"]
    if not path.is_file():
        raise FileNotFoundError(f"Thiếu ảnh testA: {path}. Kiểm tra IMAGE_DIR.")
image_transform = transforms.Compose([transforms.Resize(IMAGE_SIZE), transforms.PILToTensor()])
print(f"TestA: {len(test_samples):,} câu | Nhóm màu: {color_mask.sum():,} câu | Resize H×W: {IMAGE_SIZE}")
display(pd.DataFrame.from_dict(checkpoint_info, orient="index"))

## Tạo feature văn bản đến hết layer 10 của RoBERTa

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL, revision=TEXT_REVISION)
text_config = AutoConfig.from_pretrained(TEXT_MODEL, revision=TEXT_REVISION)
text_encoder = AutoModel.from_pretrained(
    TEXT_MODEL, revision=TEXT_REVISION, add_pooling_layer=False
)
text_encoder.encoder.layer = text_encoder.encoder.layer[:10]
text_encoder.requires_grad_(False).to(device).eval()
unique_texts = sorted({row["text"] for row in test_samples})
text_features = {}
with torch.no_grad():
    for start in tqdm(range(0, len(unique_texts), 128), desc="RoBERTa"):
        texts = unique_texts[start:start + 128]
        inputs = tokenizer(texts, padding=True, truncation=False, return_tensors="pt")
        lengths = inputs["attention_mask"].sum(dim=1).tolist()
        with torch.autocast("cuda", dtype=torch.float16):
            hidden = text_encoder(**inputs.to(device)).last_hidden_state
        hidden = hidden.to(device="cpu", dtype=torch.float16)
        for text, length, features in zip(texts, lengths, hidden):
            text_features[text] = features[:length].clone()
del text_encoder, hidden, inputs, features
gc.collect()
torch.cuda.empty_cache()

## Kiến trúc DETR + REC từ notebook huấn luyện

In [ ]:
# Copyright (c) Facebook, Inc. and its affiliates. All Rights Reserved
# Visual modules from djiajunustc/TransVG, commit:
# c8624277018fc9786a39181e229f155f933b2b59
# https://github.com/djiajunustc/TransVG/tree/main/models/visual_model
# Minimal NestedTensor utility + official backbone/PE/encoder-only execution.
import copy
import argparse
import pickle
import torchvision
from typing import Dict, List, Optional
from torch import Tensor
from torchvision.models._utils import IntermediateLayerGetter

class NestedTensor:
    def __init__(self,tensors,mask):
        self.tensors, self.mask = tensors, mask
    def decompose(self): return self.tensors,self.mask

def nested_tensor_from_tensor_list(images):
    # Tất cả ảnh trong batch cùng H,W; không padding.
    assert isinstance(images,torch.Tensor) and images.ndim == 4
    mask = torch.zeros(images.shape[0],*images.shape[-2:],device=images.device,dtype=torch.bool)
    return NestedTensor(images,mask)

class PositionEmbeddingSine(nn.Module):
    """
    This is a more standard version of the position embedding, very similar to the one
    used by the Attention is all you need paper, generalized to work on images.
    """
    def __init__(self, num_pos_feats=64, temperature=10000, normalize=False, scale=None):
        super().__init__()
        self.num_pos_feats = num_pos_feats
        self.temperature = temperature
        self.normalize = normalize
        if scale is not None and normalize is False:
            raise ValueError("normalize should be True if scale is passed")
        if scale is None:
            scale = 2 * math.pi
        self.scale = scale

    def forward(self, tensor_list: NestedTensor):
        x = tensor_list.tensors
        mask = tensor_list.mask
        assert mask is not None
        not_mask = ~mask
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)
        if self.normalize:
            eps = 1e-6
            y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
            x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale

        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)

        pos_x = x_embed[:, :, :, None] / dim_t
        pos_y = y_embed[:, :, :, None] / dim_t
        pos_x = torch.stack((pos_x[:, :, :, 0::2].sin(), pos_x[:, :, :, 1::2].cos()), dim=4).flatten(3)
        pos_y = torch.stack((pos_y[:, :, :, 0::2].sin(), pos_y[:, :, :, 1::2].cos()), dim=4).flatten(3)
        pos = torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)
        return pos

class FrozenBatchNorm2d(torch.nn.Module):
    """
    BatchNorm2d where the batch statistics and the affine parameters are fixed.

    Copy-paste from torchvision.misc.ops with added eps before rqsrt,
    without which any other models than torchvision.models.resnet[18,34,50,101]
    produce nans.
    """

    def __init__(self, n):
        super(FrozenBatchNorm2d, self).__init__()
        self.register_buffer("weight", torch.ones(n))
        self.register_buffer("bias", torch.zeros(n))
        self.register_buffer("running_mean", torch.zeros(n))
        self.register_buffer("running_var", torch.ones(n))

    def _load_from_state_dict(self, state_dict, prefix, local_metadata, strict,
                              missing_keys, unexpected_keys, error_msgs):
        num_batches_tracked_key = prefix + 'num_batches_tracked'
        if num_batches_tracked_key in state_dict:
            del state_dict[num_batches_tracked_key]

        super(FrozenBatchNorm2d, self)._load_from_state_dict(
            state_dict, prefix, local_metadata, strict,
            missing_keys, unexpected_keys, error_msgs)

    def forward(self, x):
        # move reshapes to the beginning
        # to make it fuser-friendly
        w = self.weight.reshape(1, -1, 1, 1)
        b = self.bias.reshape(1, -1, 1, 1)
        rv = self.running_var.reshape(1, -1, 1, 1)
        rm = self.running_mean.reshape(1, -1, 1, 1)
        eps = 1e-5
        scale = w * (rv + eps).rsqrt()
        bias = b - rm * scale
        return x * scale + bias

class BackboneBase(nn.Module):

    def __init__(self, name:str, backbone: nn.Module, num_channels: int, return_interm_layers: bool):
        super().__init__()
        for name, parameter in backbone.named_parameters():
            if 'layer2' not in name and 'layer3' not in name and 'layer4' not in name:
                parameter.requires_grad_(False)
        if return_interm_layers:
            return_layers = {"layer1": "0", "layer2": "1", "layer3": "2", "layer4": "3"}
        else:
            return_layers = {'layer4': "0"}
        self.body = IntermediateLayerGetter(backbone, return_layers=return_layers)
        self.num_channels = num_channels

    def forward(self, tensor_list: NestedTensor):
        xs = self.body(tensor_list.tensors)
        out: Dict[str, NestedTensor] = {}
        for name, x in xs.items():
            m = tensor_list.mask
            assert m is not None
            mask = F.interpolate(m[None].float(), size=x.shape[-2:]).to(torch.bool)[0]
            out[name] = NestedTensor(x, mask)
        return out

class Backbone(BackboneBase):
    """ResNet backbone with frozen BatchNorm."""
    def __init__(self, name: str,
                 return_interm_layers: bool,
                 dilation: bool):

        backbone = getattr(torchvision.models, name)(
            replace_stride_with_dilation=[False, False, dilation],
            weights=None, norm_layer=FrozenBatchNorm2d)
            # pretrained=is_main_process(), norm_layer=FrozenBatchNorm2d)
        assert name in ('resnet50', 'resnet101')
        num_channels = 2048
        super().__init__(name, backbone, num_channels, return_interm_layers)

class Joiner(nn.Sequential):
    def __init__(self, backbone, position_embedding):
        super().__init__(backbone, position_embedding)

    def forward(self, tensor_list: NestedTensor):
        xs = self[0](tensor_list)
        out: List[NestedTensor] = []
        pos = []
        for name, x in xs.items():
            out.append(x)
            # position encoding
            pos.append(self[1](x).to(x.tensors.dtype))

        return out, pos

class Transformer(nn.Module):

    def __init__(self, d_model=512, nhead=8, num_encoder_layers=6,
                 num_decoder_layers=6, dim_feedforward=2048, dropout=0.1,
                 activation="relu", normalize_before=False,
                 return_intermediate_dec=False):
        super().__init__()

        encoder_layer = TransformerEncoderLayer(d_model, nhead, dim_feedforward,
                                                dropout, activation, normalize_before)
        encoder_norm = nn.LayerNorm(d_model) if normalize_before else None
        self.encoder = TransformerEncoder(encoder_layer, num_encoder_layers, encoder_norm)

        if num_decoder_layers > 0:
            decoder_layer = TransformerDecoderLayer(d_model, nhead, dim_feedforward,
                                                    dropout, activation, normalize_before)
            decoder_norm = nn.LayerNorm(d_model)
            self.decoder = TransformerDecoder(decoder_layer, num_decoder_layers, decoder_norm,
                                            return_intermediate=return_intermediate_dec)
        else:
            self.decoder = None

        self._reset_parameters()

        self.d_model = d_model
        self.nhead = nhead

    def _reset_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, src, mask, pos_embed, query_embed=None):
        # flatten NxCxHxW to HWxNxC
        bs, c, h, w = src.shape
        src = src.flatten(2).permute(2, 0, 1)
        pos_embed = pos_embed.flatten(2).permute(2, 0, 1)
        mask = mask.flatten(1)
        memory = self.encoder(src, src_key_padding_mask=mask, pos=pos_embed)

        if self.decoder is not None:
            query_embed = query_embed.unsqueeze(1).repeat(1, bs, 1)
            tgt = torch.zeros_like(query_embed)
            hs = self.decoder(tgt, memory, memory_key_padding_mask=mask,
                            pos=pos_embed, query_pos=query_embed)
        
            return hs.transpose(1, 2), memory.permute(1, 2, 0).view(bs, c, h, w)
        else:
            return mask, memory

class TransformerEncoder(nn.Module):

    def __init__(self, encoder_layer, num_layers, norm=None):
        super().__init__()
        self.layers = _get_clones(encoder_layer, num_layers)
        self.num_layers = num_layers
        self.norm = norm

    def forward(self, src,
                mask: Optional[Tensor] = None,
                src_key_padding_mask: Optional[Tensor] = None,
                pos: Optional[Tensor] = None):
        output = src

        for layer in self.layers:
            output = layer(output, src_mask=mask,
                           src_key_padding_mask=src_key_padding_mask, pos=pos)

        if self.norm is not None:
            output = self.norm(output)

        return output

class TransformerEncoderLayer(nn.Module):

    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1,
                 activation="relu", normalize_before=False):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        # Implementation of Feedforward model
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        self.activation = _get_activation_fn(activation)
        self.normalize_before = normalize_before

    def with_pos_embed(self, tensor, pos: Optional[Tensor]):
        return tensor if pos is None else tensor + pos

    def forward_post(self,
                     src,
                     src_mask: Optional[Tensor] = None,
                     src_key_padding_mask: Optional[Tensor] = None,
                     pos: Optional[Tensor] = None):
        q = k = self.with_pos_embed(src, pos)
        src2 = self.self_attn(q, k, value=src, attn_mask=src_mask,
                              key_padding_mask=src_key_padding_mask)[0]
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        return src

    def forward_pre(self, src,
                    src_mask: Optional[Tensor] = None,
                    src_key_padding_mask: Optional[Tensor] = None,
                    pos: Optional[Tensor] = None):
        src2 = self.norm1(src)
        q = k = self.with_pos_embed(src2, pos)
        src2 = self.self_attn(q, k, value=src2, attn_mask=src_mask,
                              key_padding_mask=src_key_padding_mask)[0]
        src = src + self.dropout1(src2)
        src2 = self.norm2(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src2))))
        src = src + self.dropout2(src2)
        return src

    def forward(self, src,
                src_mask: Optional[Tensor] = None,
                src_key_padding_mask: Optional[Tensor] = None,
                pos: Optional[Tensor] = None):
        if self.normalize_before:
            return self.forward_pre(src, src_mask, src_key_padding_mask, pos)
        return self.forward_post(src, src_mask, src_key_padding_mask, pos)

def _get_clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for i in range(N)])

def _get_activation_fn(activation):
    """Return an activation function given a string"""
    if activation == "relu":
        return F.relu
    if activation == "gelu":
        return F.gelu
    if activation == "glu":
        return F.glu
    raise RuntimeError(F"activation should be relu/gelu, not {activation}.")

class DETR(nn.Module):
    """ This is the DETR module that performs object detection """
    def __init__(self, backbone, transformer, num_queries, train_backbone, train_transformer, aux_loss=False):
        """ Initializes the model.
        Parameters:
            backbone: torch module of the backbone to be used. See backbone.py
            transformer: torch module of the transformer architecture. See transformer.py
            num_classes: number of object classes
            num_queries: number of object queries, ie detection slot. This is the maximal number of objects
                         DETR can detect in a single image. For COCO, we recommend 100 queries.
            aux_loss: True if auxiliary decoding losses (loss at each decoder layer) are to be used.
        """
        super().__init__()
        self.num_queries = num_queries
        self.transformer = transformer
        self.backbone = backbone

        if self.transformer is not None:
            hidden_dim = transformer.d_model
            self.input_proj = nn.Conv2d(backbone.num_channels, hidden_dim, kernel_size=1)
        else:
            hidden_dim = backbone.num_channels

        if not train_backbone:
            for p in self.backbone.parameters():
                p.requires_grad_(False)
        
        if self.transformer is not None and not train_transformer:
            for m in [self.transformer, self.input_proj]:
                for p in m.parameters():
                    p.requires_grad_(False)

        self.num_channels = hidden_dim

    def forward(self, samples: NestedTensor):
        """ The forward expects a NestedTensor, which consists of:
               - samples.tensor: batched images, of shape [batch_size x 3 x H x W]
               - samples.mask: a binary mask of shape [batch_size x H x W], containing 1 on padded pixels

            It returns a dict with the following elements:
               - "pred_logits": the classification logits (including no-object) for all queries.
                                Shape= [batch_size x num_queries x (num_classes + 1)]
               - "pred_boxes": The normalized boxes coordinates for all queries, represented as
                               (center_x, center_y, height, width). These values are normalized in [0, 1],
                               relative to the size of each individual image (disregarding possible padding).
                               See PostProcess for information on how to retrieve the unnormalized bounding box.
               - "aux_outputs": Optional, only returned when auxilary losses are activated. It is a list of
                                dictionnaries containing the two above keys for each decoder layer.
        """
        if isinstance(samples, (list, torch.Tensor)):
            samples = nested_tensor_from_tensor_list(samples)
        features, pos = self.backbone(samples)

        src, mask = features[-1].decompose()
        assert mask is not None

        if self.transformer is not None:
            out = self.transformer(self.input_proj(src), mask, pos[-1], query_embed=None)
        else:
            out = [mask.flatten(1), src.flatten(2).permute(2, 0, 1)]
             
        return out

class FrozenDETRFeatures(nn.Module):
    def __init__(self,checkpoint):
        super().__init__()
        backbone = Backbone('resnet50', return_interm_layers=False, dilation=False)
        position = PositionEmbeddingSine(128, normalize=True)
        joiner = Joiner(backbone,position)
        joiner.num_channels = backbone.num_channels
        transformer = Transformer(d_model=256,nhead=8,num_encoder_layers=6,
                                  num_decoder_layers=0,dim_feedforward=2048,
                                  dropout=0.1,activation='relu',normalize_before=False)
        self.visual = DETR(joiner,transformer,num_queries=100,
                           train_backbone=False,train_transformer=False)
        # Một số checkpoint gốc chứa argparse.Namespace; chỉ allowlist lớp đó.
        with torch.serialization.safe_globals([argparse.Namespace]):
            payload = torch.load(checkpoint,map_location='cpu',weights_only=True)
        state = payload.get('model',payload)
        assert isinstance(state,dict) and state, 'Checkpoint thiếu model state_dict.'
        state = {k.removeprefix('module.'):v for k,v in state.items()}
        required = self.visual.state_dict()
        missing = sorted(set(required)-set(state))
        assert not missing, f'Checkpoint thiếu visual weights: {missing[:20]}'
        mismatched = [k for k in required if tuple(required[k].shape)!=tuple(state[k].shape)]
        assert not mismatched, f'Backbone/cấu hình checkpoint không khớp: {mismatched[:20]}'
        extra = sorted(set(state)-set(required))
        # Chỉ bỏ detection decoder/head/query; không bỏ nhầm encoder/backbone.
        allowed = ('transformer.decoder.','class_embed.','bbox_embed.','query_embed.')
        unexpected = [k for k in extra if not k.startswith(allowed)]
        assert not unexpected, f'Checkpoint có visual keys không nhận diện: {unexpected[:20]}'
        self.visual.load_state_dict({k:state[k] for k in required},strict=True)
        print('Loaded ALL backbone + input_proj + 6-layer encoder weights;',
              len(extra),'unused detection decoder/head keys.')
        del state,payload
        self.requires_grad_(False)
        self.eval()
    def train(self,mode=True):
        super().train(False)
        return self
    @torch.no_grad()
    @torch.autocast(device_type='cuda',dtype=torch.float16)
    def forward(self,images):
        padding,memory = self.visual(nested_tensor_from_tensor_list(images))
        h,w = math.ceil(images.shape[-2]/32),math.ceil(images.shape[-1]/32)
        assert tuple(memory.shape) == (h*w,len(images),256)
        assert not padding.any(), 'Ảnh đã resize cùng size nên mask phải toàn False.'
        return memory.permute(1,2,0).reshape(len(images),256,h,w).contiguous()

In [ ]:
class PositionEncoding2D(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        assert d_model % 4 == 0
        self.d_model = d_model

    def forward(self, feature_map):
        _, _, H, W = feature_map.shape
        device = feature_map.device
        n = self.d_model // 4

        frequencies = 10000 ** (
            torch.arange(n, device=device, dtype=torch.float32) / n
        )

        y = torch.linspace(0, 2 * math.pi, H, device=device)
        x = torch.linspace(0, 2 * math.pi, W, device=device)

        y = y[:, None] / frequencies[None, :]
        x = x[:, None] / frequencies[None, :]

        pos_y = torch.cat([y.sin(), y.cos()], dim=-1)
        pos_x = torch.cat([x.sin(), x.cos()], dim=-1)

        pos_y = pos_y[:, None, :].expand(H, W, -1)
        pos_x = pos_x[None, :, :].expand(H, W, -1)

        position = torch.cat([pos_y, pos_x], dim=-1)

        return position.reshape(
            1, H * W, self.d_model
        ).to(dtype=feature_map.dtype)

class DETRCachedREC(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.image_projection = nn.Conv2d(256, d_model, 1)
        self.image_position = PositionEncoding2D(d_model)
        self.text_projection = nn.Linear(768, d_model)
        self.modality_embedding = nn.Embedding(2, d_model)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=8, num_encoder_layers=6,
            num_decoder_layers=6, dim_feedforward=1024,
            dropout=0.1, activation='relu', batch_first=True,
        )
        self.object_query = nn.Embedding(1, d_model)
        self.box_head = nn.Sequential(
            nn.Linear(d_model,d_model), nn.ReLU(),
            nn.Linear(d_model,d_model), nn.ReLU(),
            nn.Linear(d_model,4), nn.Sigmoid(),
        )
    @torch.autocast(device_type='cuda', dtype=torch.float16)
    def forward(self, image_features, text_features, attention_mask):
        assert image_features.shape[1] == 256, 'Cần spatial memory DETR encoder.'
        x = image_features
        x = self.image_projection(x)
        image_tokens = x.flatten(2).transpose(1,2)
        image_tokens = image_tokens + self.image_position(x) + self.modality_embedding.weight[0]
        text_tokens = self.text_projection(text_features) + self.modality_embedding.weight[1]
        tokens = torch.cat([image_tokens, text_tokens], dim=1)
        image_padding = torch.zeros(image_features.shape[0], image_tokens.shape[1], dtype=torch.bool, device=image_features.device)
        padding = torch.cat([image_padding, ~attention_mask.bool()], dim=1)
        memory = self.transformer.encoder(tokens, src_key_padding_mask=padding)
        query = self.object_query.weight.unsqueeze(0).expand(image_features.shape[0],-1,-1)
        decoded = self.transformer.decoder(query, memory, memory_key_padding_mask=padding)
        return self.box_head(decoded[:,0,:])


class FineTuneREC(DETRCachedREC):
    def __init__(self, visual, text_config):
        super().__init__()
        self.layer4 = visual.backbone[0].body.layer4.requires_grad_(True)
        self.pos = visual.backbone[1]
        self.skip_alpha = 0.0
        self.proj = visual.input_proj.requires_grad_(True)
        self.visual_encoder = visual.transformer.encoder.requires_grad_(True)
        text = AutoModel.from_config(text_config, add_pooling_layer=False)
        self.text_tail = text.encoder.layer[-2:]

    @torch.autocast('cuda', dtype=torch.float16)
    def forward(self, image, text, attention_mask):
        x = self.proj(self.layer4(image))
        nested = nested_tensor_from_tensor_list(x)
        pos = self.pos(nested).to(x.dtype).flatten(2).permute(2,0,1)
        tokens = x.flatten(2).permute(2,0,1)
        memory = self.visual_encoder(tokens, src_key_padding_mask=nested.mask.flatten(1), pos=pos)
        x_post = memory.permute(1, 2, 0).reshape_as(x)
        image = x_post + self.skip_alpha * x if self.skip_alpha else x_post
        mask = (~attention_mask)[:,None,None,:].to(text.dtype) * torch.finfo(text.dtype).min
        for layer in self.text_tail:
            text = layer(text, attention_mask=mask)[0]
        return super().forward(image, text, attention_mask)



visual = FrozenDETRFeatures(DETR_CHECKPOINT).visual
stem = nn.Sequential(*list(visual.backbone[0].body.children())[:-1])
stem.requires_grad_(False).to(device).eval()
model = FineTuneREC(visual, text_config).requires_grad_(False).to(device).eval()
del visual

## Inference và so sánh

In [ ]:
def load_image_pixels(sample):
    with Image.open(IMAGE_DIR / sample['file_name']) as image:
        assert image.size == (sample['width'], sample['height']), 'Metadata ảnh sai.'
        return image_transform(image.convert('RGB'))



class TestDataset(Dataset):
    def __len__(self):
        return len(test_samples)

    def __getitem__(self, index):
        row = test_samples[index]
        x, y, w, h = row["bbox"]
        width, height = row["width"], row["height"]
        box = torch.tensor(
            [(x + w / 2) / width, (y + h / 2) / height, w / width, h / height],
            dtype=torch.float32,
        )
        return load_image_pixels(row), text_features[row["text"]], box, color_mask[index].item()


def collate(batch):
    images, texts, boxes, has_color = zip(*batch)
    lengths = torch.tensor([len(t) for t in texts])
    text = nn.utils.rnn.pad_sequence(texts, batch_first=True)
    mask = torch.arange(text.shape[1])[None,:] < lengths[:,None]
    return dict(images=torch.stack(images), text_features=text,
                attention_mask=mask, boxes=torch.stack(boxes),
                has_color=torch.tensor(has_color, dtype=torch.bool))


test_loader = DataLoader(
    TestDataset(), batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate,
    persistent_workers=NUM_WORKERS > 0,
)
@torch.no_grad()
def matched_iou(pred, target):
    # IoU từng cặp, không tạo ma trận B×B rồi lấy đường chéo.
    p = box_convert(pred.float(), in_fmt='cxcywh', out_fmt='xyxy')
    t = box_convert(target.float(), in_fmt='cxcywh', out_fmt='xyxy')
    wh = (torch.minimum(p[:,2:],t[:,2:])-torch.maximum(p[:,:2],t[:,:2])).clamp(min=0)
    intersection = wh.prod(-1)
    union = (p[:,2:]-p[:,:2]).clamp(min=0).prod(-1) + (t[:,2:]-t[:,:2]).clamp(min=0).prod(-1) - intersection
    return intersection / union.clamp(min=1e-7)


image_mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
image_std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)


@torch.no_grad()
def evaluate(name, path):
    checkpoint = torch.load(path, map_location="cpu", weights_only=True, mmap=True)
    cfg = check_checkpoint(name, checkpoint)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.skip_alpha = float(cfg.get("skip_alpha", 0.0))
    del checkpoint

    ious, masks = [], []
    for batch in tqdm(test_loader, desc=name):
        images = batch["images"].to(device, non_blocking=True).float().div_(255)
        images = images.sub_(image_mean).div_(image_std).half()
        with torch.autocast("cuda", dtype=torch.float16):
            image_features = stem(images).half()
        pred = model(
            image_features,
            batch["text_features"].to(device, non_blocking=True),
            batch["attention_mask"].to(device, non_blocking=True),
        )
        boxes = batch["boxes"].to(device, non_blocking=True)
        ious.append(matched_iou(pred, boxes).cpu().numpy())
        masks.append(batch["has_color"].numpy())
    values = np.concatenate(ious)
    if not np.array_equal(np.concatenate(masks), color_mask) or len(values) != len(test_samples):
        raise RuntimeError("Kết quả lệch thứ tự hoặc thiếu mẫu testA.")
    if not np.isfinite(values).all():
        raise RuntimeError(f"{name}: IoU có NaN hoặc Inf.")
    return values


test_iou = {name: evaluate(name, path) for name, path in checkpoint_paths.items()}

In [ ]:
metric_rows = []
for group, mask in [("Toàn bộ testA", np.ones(len(test_samples), dtype=bool)),
                    ("TestA · nhóm màu", color_mask)]:
    baseline = test_iou["DETR gốc"][mask]
    skip = test_iou["DETR + skip"][mask]
    base_acc = float((baseline >= 0.5).mean() * 100)
    skip_acc = float((skip >= 0.5).mean() * 100)
    base_miou = float(baseline.mean(dtype=np.float64))
    skip_miou = float(skip.mean(dtype=np.float64))
    metric_rows.append({
        "Tập": group,
        "Số câu": int(mask.sum()),
        "DETR gốc Acc@0.5 (%)": base_acc,
        "DETR + skip Acc@0.5 (%)": skip_acc,
        "Δ Acc (điểm %)": skip_acc - base_acc,
        "DETR gốc mIoU": base_miou,
        "DETR + skip mIoU": skip_miou,
        "Δ mIoU": skip_miou - base_miou,
    })

table = pd.DataFrame(metric_rows).set_index("Tập")
display(table.round(4))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
table.to_csv(OUTPUT_DIR / "testA_comparison.csv")
(OUTPUT_DIR / "testA_comparison.json").write_text(
    json.dumps(
        {"checkpoints": checkpoint_info, "color_words": COLOR_WORDS, "metrics": metric_rows},
        ensure_ascii=False, indent=2,
    ),
    encoding="utf-8",
)
print("Kết quả:", OUTPUT_DIR)